# NumPy Core

This notebook covers only the `NumPy` topics that are essential for later `PyTorch` work. It does not try to be exhaustive.

Two habits matter most in this notebook:

1. Whenever you see array operations, check the `shape` first.
2. Prefer vectorized thinking whenever possible.

## Learning Goals

After this notebook, you should be able to:

1. Create and inspect `ndarray` objects.
2. Use indexing, slicing, and boolean indexing.
3. Understand the basic rules of broadcasting.
4. Rewrite simple loops as vectorized expressions.
5. Handle `reshape`, adding axes, and transpose correctly.
6. Transfer these skills to later `PyTorch Tensor` work.

In [ ]:
import numpy as np

## `ndarray` Basics

The core object in NumPy is the `ndarray`. You can think of it as a rectangular block of values with a known structure. The values matter, but the structure matters just as much: `shape` tells you how many values exist along each axis, `ndim` tells you how many axes there are, and `dtype` tells you what type of value is stored.

This is directly relevant to PyTorch. Most tensor debugging later starts with the same questions: what is the shape, how many dimensions does it have, and is the dtype what the next operation expects?

In [ ]:
a = np.array([1, 2, 3], dtype=np.float32)
b = np.zeros((2, 3))
c = np.arange(12).reshape(3, 4)

print("a =", a)
print("a.shape / a shape =", a.shape)
print("a.dtype / a dtype =", a.dtype)
print("a.ndim / a ndim =", a.ndim)
print()
print("b =")
print(b)
print("b.shape =", b.shape)
print()
print("c =")
print(c)
print("c.shape =", c.shape)

You should become sensitive not only to the values in an array, but also to its structure. The length of each axis, the total number of dimensions, and the element dtype determine whether later operations will work. When model code fails, checking `shape` and `dtype` is often the fastest first debugging step.

In [ ]:
# Exercise 1
#
# Create an array named arr.
#
# What to build:
# - Use np.arange to create the numbers 0 through 11.
# - Store them as float32 values, not int64.
# - Reshape the result to shape (3, 4).
#
# What to inspect:
# - print the full array
# - print arr.shape, which should be (3, 4)
# - print arr.dtype, which should be float32
# - print the last column, which is arr[:, -1]

arr = None

# print(arr)
# print(arr.shape)
# print(arr.dtype)
# print(arr[:, -1])

In [ ]:
# Exercise 1 Reference Solution

arr = np.arange(12, dtype=np.float32).reshape(3, 4)
print(arr)
print(arr.shape)
print(arr.dtype)
print(arr[:, -1])

## Indexing, Slicing, and Boolean Indexing

Indexing is how you select specific rows, columns, or elements. Slicing such as `grid[1:4, 1:4]` selects a rectangular region. Integer-array indexing such as `grid[[0, 2, 4]]` selects specific rows. Boolean indexing such as `grid[grid > 10]` keeps only elements where a condition is true.

These operations transfer directly to tensor work later. In real ML code, you use the same ideas to select feature columns, take a subset of samples from a batch, or filter rows that match a condition.

In [ ]:
matrix = np.arange(1, 17).reshape(4, 4)

print("matrix =")
print(matrix)
print()
print("middle two rows of the first two columns / middle two rows of first two columns:")
print(matrix[1:3, :2])
print()
print("last column / last column:")
print(matrix[:, -1])
print()
mask = matrix % 2 == 0
print("boolean masks / boolean mask:")
print(mask)
print("all even numbers / all even numbers:")
print(matrix[mask])

In [ ]:
# Exercise 2
#
# Work with the 5x5 array below.
#
# Fill in these four variables:
# - center: the center 3x3 block, using rows 1 through 3 and columns 1 through 3.
# - picked_rows: rows 0, 2, and 4.
# - bigger_than_10: all elements whose value is greater than 10.
# - reversed_rows: the same grid with row order reversed.
#
# The goal is to practice four different selection styles: slicing, integer row
# selection, boolean masking, and reverse slicing.

grid = np.arange(25).reshape(5, 5)

# center =
# picked_rows =
# bigger_than_10 =
# reversed_rows =

# print(grid)
# print(center)
# print(picked_rows)
# print(bigger_than_10)
# print(reversed_rows)

In [ ]:
# Exercise 2 Reference Solution

grid = np.arange(25).reshape(5, 5)
center = grid[1:4, 1:4]
picked_rows = grid[[0, 2, 4]]
bigger_than_10 = grid[grid > 10]
reversed_rows = grid[::-1]

print(grid)
print(center)
print(picked_rows)
print(bigger_than_10)
print(reversed_rows)

## Broadcasting

Broadcasting is NumPy's rule for combining arrays with different but compatible shapes. NumPy compares shapes from the last dimension backward. At each position, the sizes must either be equal or one of them must be `1`. If those rules hold, NumPy behaves as if the smaller array were expanded without physically copying all values.

This matters because many ML operations rely on broadcasting. Subtracting a column mean from a full feature matrix, scaling each feature by its standard deviation, or adding a bias vector to a batch of predictions all use the same idea.

In [ ]:
features = np.array([
    [1.0, 10.0],
    [2.0, 20.0],
    [3.0, 30.0],
])

offset = np.array([100.0, 1000.0])
scale = np.array([[1.0], [10.0], [100.0]])

print("features + offset =")
print(features + offset)
print()
print("features * scale =")
print(features * scale)

You should be able to explain the results above using `shape`.

- `features.shape == (3, 2)`
- broadcasts by row

In [ ]:
# Exercise 3
#
# Implement center_by_column(x).
#
# Input:
# - x is a 2D NumPy array with shape (num_rows, num_columns).
#
# What the function should do:
# - Compute the mean of each column.
# - Subtract each column's mean from that column.
# - Return the centered array.
#
# Shape requirement:
# - Use keepdims=True or an equivalent shape so the column means broadcast
#   correctly against x.

def center_by_column(x):
    # TODO
    pass


# x = np.array([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
# print(center_by_column(x))

In [ ]:
# Exercise 3 Reference Solution

def center_by_column_solution(x):
    col_mean = x.mean(axis=0, keepdims=True)
    return x - col_mean


x = np.array([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
print(center_by_column_solution(x))

## Vectorization

Vectorization means writing array operations that work on many values at once instead of looping over individual elements in Python. The result is usually shorter and closer to the mathematical expression. It is also often faster because NumPy can run the underlying operation in optimized compiled code.

For learning, the main goal is not just speed. The main goal is to recognize when an operation such as `(pred - target) ** 2` describes the whole array calculation directly.

In [ ]:
values = np.arange(1, 6)

loop_square_sum = 0.0
for value in values:
    loop_square_sum += value ** 2

vectorized_square_sum = (values ** 2).sum()

print("loop version / loop version:", loop_square_sum)
print("vectorized version / vectorized version:", vectorized_square_sum)

In [ ]:
# Exercise 4
#
# Implement mean squared error using vectorized NumPy operations.
#
# Formula:
# mse = mean((pred - target) ** 2)
#
# What to return:
# - a single scalar number
# - do not write a Python for-loop over the elements

pred = np.array([2.5, 0.0, 2.0, 8.0])
target = np.array([3.0, -0.5, 2.0, 7.0])


def mse_vectorized(pred, target):
    # TODO
    pass


# print(mse_vectorized(pred, target))

In [ ]:
# Exercise 4 Reference Solution

def mse_vectorized_solution(pred, target):
    return np.mean((pred - target) ** 2)


print(mse_vectorized_solution(pred, target))

## Shape Transformations

Shape transformations change how the same values are arranged or interpreted. `reshape` changes the visible shape while keeping the same number of elements. `transpose` changes axis order. Adding or removing axes changes how later operations broadcast.

When you transform shapes, always ask which axis means sample, height, width, channel, or feature. Most shape bugs come from having the right number of values but the wrong axis meaning.

In [ ]:
flat = np.arange(24)
cube = flat.reshape(2, 3, 4)
transposed = cube.transpose(0, 2, 1)
expanded = np.arange(6).reshape(2, 3)[:, np.newaxis, :]

print("flat.shape =", flat.shape)
print("cube.shape =", cube.shape)
print("transposed.shape =", transposed.shape)
print("expanded.shape =", expanded.shape)

In [ ]:
# Exercise 5
#
# flat_images has shape (2, 12). Treat it as two flattened images.
#
# Step 1:
# - Reshape flat_images into images with shape (2, 3, 4).
# - This means 2 images, each with height 3 and width 4.
#
# Step 2:
# - Transpose images into images_t with shape (2, 4, 3).
# - Keep the image axis first, but swap height and width.

flat_images = np.arange(24).reshape(2, 12)

# images =
# images_t =

# print(images.shape)
# print(images_t.shape)

In [ ]:
# Exercise 5 Reference Solution

flat_images = np.arange(24).reshape(2, 12)
images = flat_images.reshape(2, 3, 4)
images_t = images.transpose(0, 2, 1)

print(images.shape)
print(images_t.shape)

## A Small ML Scenario

Below is a tiny tabular example to illustrate feature standardization.


In [ ]:
# columns: hours, attendance, sleep, passed
raw = np.array([
    [1.5, 0.70, 6.0, 0],
    [3.0, 0.90, 7.0, 1],
    [2.2, 0.80, 6.5, 1],
    [1.0, 0.60, 5.5, 0],
], dtype=np.float32)

features = raw[:, :-1]
labels = raw[:, -1].astype(np.int64)

feature_mean = features.mean(axis=0, keepdims=True)
feature_std = features.std(axis=0, keepdims=True)
features_std = (features - feature_mean) / (feature_std + 1e-8)

print("features.shape =", features.shape)
print("labels.shape =", labels.shape)
print("standardized features / standardized features =")
print(features_std)

In [ ]:
# Exercise 6
#
# Implement split_xy_and_standardize(raw).
#
# Input:
# - raw is a 2D array.
# - The last column is the label.
# - Every earlier column is a numeric feature.
#
# Output:
# - x_std: standardized features, where each feature column has approximately
#   mean 0 and standard deviation 1.
# - y: labels from the last column, converted to int64.
#
# Implementation steps:
# - Split x from raw[:, :-1].
# - Split y from raw[:, -1].
# - Compute column means and column standard deviations from x.
# - Return (x - mean) / (std + 1e-8), y.

def split_xy_and_standardize(raw):
    # TODO
    pass


# x_std, y = split_xy_and_standardize(raw)
# print(x_std)
# print(y)

In [ ]:
# Exercise 6 Reference Solution

def split_xy_and_standardize_solution(raw):
    x = raw[:, :-1]
    y = raw[:, -1].astype(np.int64)
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    x_std = (x - mean) / (std + 1e-8)
    return x_std, y


x_std, y = split_xy_and_standardize_solution(raw)
print(x_std)
print(y)

## Debugging Task

The following code fails because of a broadcasting error.

You should first decide whether the intention is row-wise scaling or column-wise scaling.


In [ ]:
x = np.arange(12).reshape(3, 4)
weights = np.array([0.1, 0.2, 0.3])

# print(x * weights)
# TODO:
# fix it for column-wise scaling

In [ ]:
# Debugging Task Reference Solution

x = np.arange(12).reshape(3, 4)
weights = np.array([0.1, 0.2, 0.3])

row_scaled = x * weights[:, np.newaxis]
print("row-wise scaling / row-wise scaling =")
print(row_scaled)
print()

col_weights = np.array([0.1, 0.2, 0.3, 0.4])
col_scaled = x * col_weights
print("column-wise scaling / column-wise scaling =")
print(col_scaled)

## Summary

The most important takeaway from this notebook is `shape` thinking.

You should now be able to answer:

1. What do `shape`, `ndim`, and `dtype` each represent?
2. What is the difference between boolean indexing and normal slicing?
3. What is the minimal rule for broadcasting?
4. Why is vectorization usually better for numerical computing?
5. What is the difference between `reshape` and `transpose`?

Suggested next step:

- Move to the Pandas notebook and connect array thinking to tabular data work.